# How Evals Actually Work — A Practical Guide

## The simplest eval that actually works

You need three things:

1. A question
2. An expected answer (or a way to check correctness)
3. A loop that runs it multiple times

## The test data

These are 6 forensic questions from the BOTSv3 Incident B2 (AWS Key Compromise), each with a known correct answer.

**Running in Colab instead?** Upload this notebook to [colab.research.google.com](https://colab.research.google.com), run the Colab cell below (it clones the repo and installs louie-py), and set `IS_COLAB = True` in the settings cell. Colab has no `.env`, so the Louie setup cell further down will ask for your Graphistry Hub login.


In [ ]:
# Colab only: run this cell once, then set IS_COLAB = True in the settings cell below.
# Local JupyterLab: skip it.
import os
if not os.path.isdir("bsides-augusta-2026"):
    !git clone -q https://github.com/graphistry/bsides-augusta-2026.git
%pip install -q louieai graphistry


In [ ]:
# Load the canonical question set -- single source of truth.
# (Previously inlined here, which drifted from lab2/botsv3_b2_tests.json.)
import json, os
from pathlib import Path

# ---- Settings -------------------------------------------------------------
IS_COLAB = False              # Swap to True if you are running in Colab
# ---------------------------------------------------------------------------
LAB_DIR = Path("./bsides-augusta-2026/labs/lab3") if IS_COLAB else Path(".")

TESTS_PATH = Path(os.environ.get("B2_TESTS_PATH", str(LAB_DIR / "../lab2/botsv3_b2_tests.json")))
with TESTS_PATH.open() as f:
    B2_TESTS = json.load(f)

print(f"{len(B2_TESTS)} questions loaded from {TESTS_PATH}")
for qid, q in B2_TESTS.items():
    print(f"  [{q['difficulty']:>6}] {q['name']}")


## Answer checking

Three levels, from simple to sophisticated:

- **Level 1: Substring match** — `"5244329601" in agent_output` — works for factual answers with a unique string
- **Level 2: Regex pattern** — `re.search(r"xenial\s+xerus", output, re.IGNORECASE)` — works when casing or formatting varies
- **Level 3: LLM-as-judge** — ask another LLM "is this correct?" — works for open-ended answers, but the judge can be wrong too

In [ ]:
import re

def check_answer(output: str, patterns: list[str]) -> bool:
    """Check if output matches any expected pattern (Level 1 + 2)."""
    for pattern in patterns:
        if pattern.startswith("re:"):
            if re.search(pattern[3:], output, re.IGNORECASE):
                return True
        elif pattern.lower() in output.lower():
            return True
    return False

# Test it
print(check_answer("The case ID is 5244329601.", ["5244329601"]))        # True
print(check_answer("The codename is Xenial Xerus.", ["re:(?i)xenial\\s+xerus"]))  # True
print(check_answer("I could not find the answer.", ["5244329601"]))      # False

## Why you run it multiple times

This is the big difference from unit tests. If you run it once and it passes, you don't know if you got lucky. Run it 3-5 times:

```
Run 0: PASS — got '5244329601'
Run 1: PASS — got '5244329601'
Run 2: FAIL — got 'I could not find the support case'

Pass rate: 67% over 3 runs
```

Now you know something useful: it works most of the time but not always. A unit test is pass/fail. An eval gives you a **rate**.

## Setup: Authenticate with louie-py

In [ ]:
import os
import time
from pathlib import Path
from louieai._client import LouieClient
from louieai.notebook import Cursor

# Load .env file from repo root
env_path = Path(os.environ.get("LAB_ENV_FILE", str(LAB_DIR / "../../.env")))  # LAB_DIR from the settings cell above
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

# --- Authentication ---
LOUIE_HOST = os.environ.get("LOUIE_HOST", "https://den.louie.ai")
GRAPHISTRY_HOST = os.environ.get("GRAPHISTRY_HOST", "hub.graphistry.com")

CLIENT_KWARGS = dict(timeout=600.0, streaming_timeout=600.0)

if os.environ.get("GRAPHISTRY_HUB_KEY_ID"):
    louie_client = LouieClient(
        server_url=LOUIE_HOST,
        personal_key_id=os.environ["GRAPHISTRY_HUB_KEY_ID"],
        personal_key_secret=os.environ["GRAPHISTRY_HUB_KEY_SECRET"],
        graphistry_server=GRAPHISTRY_HOST,
        **CLIENT_KWARGS,
    )
elif os.environ.get("GRAPHISTRY_USERNAME"):
    louie_client = LouieClient(
        server_url=LOUIE_HOST,
        username=os.environ["GRAPHISTRY_USERNAME"],
        password=os.environ["GRAPHISTRY_PASSWORD"],
        graphistry_server=GRAPHISTRY_HOST,
        **CLIENT_KWARGS,
    )
else:
    # No .env (for example in Colab): ask for the Graphistry Hub login, which Louie uses too
    from getpass import getpass
    print("No credentials found in .env or the environment.")
    os.environ["GRAPHISTRY_USERNAME"] = input("Graphistry Hub username: ").strip()
    os.environ["GRAPHISTRY_PASSWORD"] = getpass("Graphistry Hub password: ")
    louie_client = LouieClient(
        server_url=LOUIE_HOST,
        username=os.environ["GRAPHISTRY_USERNAME"],
        password=os.environ["GRAPHISTRY_PASSWORD"],
        graphistry_server=GRAPHISTRY_HOST,
        **CLIENT_KWARGS,
    )
    print("Authenticated with username/password (prompted).")

lui = Cursor(client=louie_client, folder="training-lab3-evals")
print("louie-py ready.")

## Run the eval: single question, 3 times

Start simple — one question, run it 3 times, see the pass rate.

In [ ]:
q = B2_TESTS["b2_q19_support_case_id"]
print(f"Question: {q['name']}")
print(f"Expected: {q['expected_output']}\n")

results = []
for run in range(3):
    lui.new(name=f"Eval q19 run {run}")  # Fresh thread each time
    start = time.time()
    lui(q["question"], agent="SplunkAgent")
    elapsed = time.time() - start

    answer = lui.text or ""
    correct = check_answer(answer, q["expected_patterns"])
    results.append({"run": run, "correct": correct, "elapsed": elapsed, "answer": answer[:100]})
    print(f"  Run {run}: {'PASS' if correct else 'FAIL'} ({elapsed:.1f}s)")

pass_rate = sum(r["correct"] for r in results) / len(results)
print(f"\nPass rate: {pass_rate:.0%}")

## Run the eval: all questions

Now run all 6 questions. For speed, we'll do 1 run per question first (increase `NUM_RUNS` for more confidence).

In [ ]:
NUM_RUNS = 1  # Change to 3 for a real eval (takes longer)

all_results = []

for qid, q in B2_TESTS.items():
    print(f"\n--- {q['name']} [{q['difficulty']}] ---")
    passes = 0
    times = []

    for run in range(NUM_RUNS):
        lui.new(name=f"Eval {qid} run {run}")
        start = time.time()
        lui(q["question"], agent="SplunkAgent")
        elapsed = time.time() - start
        times.append(elapsed)

        answer = lui.text or ""
        correct = check_answer(answer, q["expected_patterns"])
        if correct:
            passes += 1
        print(f"  Run {run}: {'PASS' if correct else 'FAIL'} ({elapsed:.1f}s)")

    all_results.append({
        "id": qid,
        "name": q["name"],
        "difficulty": q["difficulty"],
        "expected": q["expected_output"],
        "passes": passes,
        "runs": NUM_RUNS,
        "pass_rate": passes / NUM_RUNS,
        "avg_time": sum(times) / len(times),
    })
    print(f"  Pass rate: {passes}/{NUM_RUNS}")

## Results: the spreadsheet

A "proper" eval looks like this — not a framework, just a table.

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)
df["pass_rate_str"] = df["pass_rate"].apply(lambda x: f"{x:.0%}")
df["avg_time_str"] = df["avg_time"].apply(lambda x: f"{x:.1f}s")

display(df[["name", "difficulty", "passes", "runs", "pass_rate_str", "avg_time_str"]]
        .rename(columns={
            "name": "Question",
            "difficulty": "Difficulty",
            "passes": "Passes",
            "runs": "Runs",
            "pass_rate_str": "Pass Rate",
            "avg_time_str": "Avg Time",
        }))

## Reading the results

This tells you:
- **What's reliable** (100% pass rate — ship it)
- **What's flaky** (33-67% — needs a skill to stabilize it)
- **What's broken** (0% — needs a different approach entirely)
- **What's slow** (high avg time — too many queries)

In [ ]:
# Summary by difficulty
for difficulty in ["easy", "medium", "hard"]:
    subset = [r for r in all_results if r["difficulty"] == difficulty]
    if subset:
        total_passes = sum(r["passes"] for r in subset)
        total_runs = sum(r["runs"] for r in subset)
        avg_time = sum(r["avg_time"] for r in subset) / len(subset)
        print(f"  {difficulty:>6}: {total_passes}/{total_runs} pass rate, {avg_time:.1f}s avg")

## The eval loop

The eval isn't a one-time check. It's a loop:

```
1. Run eval → "33% pass rate on IAM key question"
2. Diagnose why → "agent uses dc(errorCode) instead of dc(errorMessage)"
3. Fix it → add guidance to skill: "use dc(errorMessage) for distinct errors"
4. Run eval again → "100% pass rate on IAM key question"
5. Check you didn't break anything else → run all questions
```

This is the same red-green-refactor loop as TDD, just with probabilities instead of pass/fail.

## Level 3: LLM-as-judge

For questions where substring matching isn't enough (open-ended answers, narratives), you can ask louie itself to judge the answer.

In [ ]:
# Run a question first
q = B2_TESTS["b2_q18_iam_key_errors"]
lui.new(name="LLM-judge demo - investigation")
lui(q["question"], agent="SplunkAgent")
agent_answer = lui.text or ""
print(f"Agent said: {agent_answer[:200]}...")

In [ ]:
# Now ask louie to judge the answer
lui.new(name="LLM-judge demo - judgment")
lui(f"""The question was: {q['question']}
The expected answer is: {q['expected_output']}
The agent said: {agent_answer}

Is the agent's answer correct? Reply YES or NO.
If NO, classify the error:
- WRONG_ANSWER: agent gave a specific but incorrect answer
- HALLUCINATED: agent made up data not in the results
- WRONG_PATH: agent searched the wrong sourcetype or field
- TIMEOUT: agent didn't finish
""")

judgment = lui.text or ""
print(f"Judge says:\n{judgment}")

## The honest summary

| What you can automate today | What still needs a human |
|---------------------------|------------------------|
| Run the same questions N times | Decide which questions matter |
| Check answers with substring/regex | Evaluate quality of reasoning |
| Measure speed and tool call count | Judge if the SPL was elegant |
| Track pass rates over time | Decide when "good enough" is good enough |
| Use LLM-as-judge for open-ended answers | Validate the judge isn't wrong |

Start with the simple version (substring match, 3 runs, a table). You can get fancier later, but most teams never need to.